# Tutorial 5: From Black-Scholes to Quantum Amplitude Estimation

This notebook walks through the progression from the analytical Black-Scholes formula
to Monte Carlo simulation to quantum amplitude estimation (QAE) for European option pricing.

**Reference**: Stamatopoulos et al. (2020), Woerner & Egger (2019).

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Black-Scholes Analytical Price

The closed-form solution for a European call under GBM.

In [ ]:
from qufin.options.classical.black_scholes import bs_price, bs_greeks

S, K, sigma, r, T = 100, 105, 0.2, 0.05, 1.0

call_price = bs_price(s=S, k=K, sigma=sigma, r=r, T=T, option_type="call")
put_price = bs_price(s=S, k=K, sigma=sigma, r=r, T=T, option_type="put")

print(f"European Call: ${call_price:.4f}")
print(f"European Put:  ${put_price:.4f}")
print(f"Put-Call Parity check: C - P = {call_price - put_price:.4f}, S - K*e^(-rT) = {S - K * np.exp(-r * T):.4f}")

## 2. Greeks

In [ ]:
greeks = bs_greeks(s=S, k=K, sigma=sigma, r=r, T=T)

print(f"Delta: {greeks.delta:.4f}")
print(f"Gamma: {greeks.gamma:.4f}")
print(f"Vega:  {greeks.vega:.4f}")
print(f"Theta: {greeks.theta:.4f}")
print(f"Rho:   {greeks.rho:.4f}")

## 3. Classical Monte Carlo

Monte Carlo estimates the expected discounted payoff by simulating many price paths.

In [ ]:
from qufin.options.classical.monte_carlo import european_mc

mc_price = european_mc(
    s=S, k=K, sigma=sigma, r=r, T=T,
    n_paths=100_000,
    option_type="call",
)

print(f"Monte Carlo price: ${mc_price:.4f}")
print(f"Black-Scholes:     ${call_price:.4f}")
print(f"Error:             ${abs(mc_price - call_price):.4f}")

## 4. MC Convergence

Classical MC converges as $O(1/\sqrt{N})$. QAE achieves $O(1/N)$ -- a quadratic speedup.

In [ ]:
path_counts = [100, 1000, 10_000, 100_000]

for n in path_counts:
    price = european_mc(s=S, k=K, sigma=sigma, r=r, T=T, n_paths=n, option_type="call")
    err = abs(price - call_price)
    print(f"  N={n:>7d}: price=${price:.4f}  error=${err:.4f}")

## 5. Quantum Amplitude Estimation

QAE encodes the payoff distribution into quantum amplitudes and uses
amplitude estimation to extract the expected value.

In [ ]:
from qufin.options.amplitude_estimation.european_qae import (
    EuropeanQAESpec, build_european_estimation_problem,
)
from qufin.options.amplitude_estimation.iqae import (
    IQAEConfig, IterativeAmplitudeEstimation,
)
from qufin.backends.qiskit_backend import QiskitAerBackend

backend = QiskitAerBackend(shots=4096)

spec = EuropeanQAESpec(
    s=S, k=K, sigma=sigma, r=r, T=T,
    n_qubits=5,
    option_type="call",
)
problem = build_european_estimation_problem(spec)

iqae = IterativeAmplitudeEstimation(
    problem=problem,
    backend=backend,
    config=IQAEConfig(epsilon_target=0.01),
)
qae_result = iqae.estimate()

print(f"IQAE price:      ${qae_result.value:.4f}")
print(f"Black-Scholes:   ${call_price:.4f}")
print(f"Error:           ${abs(qae_result.value - call_price):.4f}")

## 6. Put Option via QAE

In [ ]:
spec_put = EuropeanQAESpec(
    s=S, k=K, sigma=sigma, r=r, T=T,
    n_qubits=5,
    option_type="put",
)
problem_put = build_european_estimation_problem(spec_put)
iqae_put = IterativeAmplitudeEstimation(
    problem=problem_put, backend=backend,
    config=IQAEConfig(epsilon_target=0.01),
)
put_result = iqae_put.estimate()

print(f"IQAE put price:  ${put_result.value:.4f}")
print(f"BS put price:    ${put_price:.4f}")

## Summary

In this tutorial we covered:
- Black-Scholes analytical pricing and Greeks
- Classical Monte Carlo and its $O(1/\sqrt{N})$ convergence
- Quantum amplitude estimation achieving the same result
- The theoretical $O(1/N)$ quantum speedup

**Next**: Tutorial 06 compares all four QAE variants (canonical, IQAE, MLAE, FQAE).